# VoltRelay Battery Swapping Analytics
## End-to-End Energy Data Analytics & Business Intelligence

**Author:** Singipurapu Dinesh  
**Analysis period:** January 2024 – June 2025  
**Tools:** Python, Pandas, NumPy, Plotly, Jupyter Notebook

### Purpose
This notebook is the presentation-ready analytics layer for the VoltRelay battery-swapping project. It uses the project's generated analysis tables where available and adds validation, geography, queue experience, battery lifecycle, efficiency, recommendations, and limitations.

> **Important:** The dataset is synthetic. Completed swap charges are treated as revenue, not profit. Rider inactivity is a churn proxy rather than confirmed permanent churn. Descriptive relationships do not prove causation.


## 1. Project overview and business objectives

### Business questions
1. How is the network performing over time?
2. Which cities and stations contribute the most activity and revenue?
3. Where are failures concentrated?
4. What does queue experience look like for riders?
5. How healthy are the batteries and how does health vary by pack type, supplier, or manufacturing lot?
6. How are riders engaging, retaining, and becoming inactive?
7. What operational and financial-efficiency signals can be measured from the available data?
8. What evidence-based actions should operations teams investigate next?

### Notebook design
The notebook separates **descriptive evidence** from recommendations. Missing optional source tables are reported rather than silently replaced.


In [24]:
# 2. Imports and project-path discovery
from pathlib import Path
import warnings
import re
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)

candidate_roots = [
    Path(r"C:\VoltRelay-Battery-Swapping-Analytics"),
    Path.cwd(),
    Path.cwd().parent,
]
PROJECT_ROOT = next(
    (p for p in candidate_roots if (p / "data").exists()),
    Path.cwd().parent
)

TABLES_DIR = PROJECT_ROOT / "data" / "outputs" / "tables"
FINDINGS_DIR = PROJECT_ROOT / "data" / "outputs" / "findings"
FIGURES_DIR = PROJECT_ROOT / "data" / "outputs" / "figures"
OUTPUT_DIR = PROJECT_ROOT / "data" / "outputs"

print("Project root:", PROJECT_ROOT)
print("Tables directory:", TABLES_DIR, "| exists:", TABLES_DIR.exists())
print("Outputs directory:", OUTPUT_DIR, "| exists:", OUTPUT_DIR.exists())


Project root: C:\VoltRelay-Battery-Swapping-Analytics
Tables directory: C:\VoltRelay-Battery-Swapping-Analytics\data\outputs\tables | exists: True
Outputs directory: C:\VoltRelay-Battery-Swapping-Analytics\data\outputs | exists: True


In [25]:
# 3. Reusable helpers
def load_table(*names, required=False):
    for name in names:
        path = TABLES_DIR / name
        if path.exists():
            df = pd.read_csv(path)
            print(f"Loaded {name}: {len(df):,} rows")
            return df
    if required:
        raise FileNotFoundError(f"None of these files found: {names}")
    print("Not found:", names)
    return pd.DataFrame()

def find_col(df, candidates):
    if df is None or df.empty:
        return None
    lower = {str(c).lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in lower:
            return lower[candidate.lower()]
    for candidate in candidates:
        for col in df.columns:
            if candidate.lower() in str(col).lower():
                return col
    return None

def numeric_col(df, candidates):
    col = find_col(df, candidates)
    if col is None:
        return None
    s = pd.to_numeric(df[col], errors="coerce")
    return col if s.notna().any() else None

def last_row(df, date_candidates=("month", "year_month", "date", "month_start")):
    if df is None or df.empty:
        return None
    dc = find_col(df, date_candidates)
    if dc:
        temp = df.copy()
        temp["_date"] = pd.to_datetime(temp[dc], errors="coerce")
        if temp["_date"].notna().any():
            return temp.sort_values("_date").iloc[-1]
    return df.iloc[-1]

def value_from(df, candidates, row=None):
    col = find_col(df, candidates)
    row = row if row is not None else last_row(df)
    if col is None or row is None:
        return np.nan
    return pd.to_numeric(pd.Series([row[col]]), errors="coerce").iloc[0]

def pct(numerator, denominator):
    numerator = pd.to_numeric(numerator, errors="coerce")
    denominator = pd.to_numeric(denominator, errors="coerce")
    return np.where(denominator != 0, numerator / denominator * 100, np.nan)

def discover_csv(filename):
    candidates = [
        PROJECT_ROOT / filename,
        PROJECT_ROOT / "data" / filename,
        PROJECT_ROOT / "data" / "raw" / filename,
        PROJECT_ROOT / "data" / "processed" / filename,
        TABLES_DIR / filename,
    ]
    for p in candidates:
        if p.exists():
            return p
    return None

def load_optional_raw(*names):
    for name in names:
        path = discover_csv(name)
        if path:
            try:
                df = pd.read_csv(path)
                print(f"Loaded optional source {path.relative_to(PROJECT_ROOT)}: {len(df):,} rows")
                return df
            except Exception as exc:
                print(f"Could not read {path}: {exc}")
    return pd.DataFrame()


## 2. Dataset description and data quality

The first validation layer checks the analysis tables and any core source files that are present in the project. The goal is to make data-quality assumptions visible before interpreting charts.

Checks include:
- row counts and missing values;
- duplicate rows and duplicate likely IDs;
- reference integrity where matching keys are available;
- invalid negative telemetry/odometer values;
- timestamp parsing and future timestamps;
- known firmware timestamp anomalies when a firmware-related field is present.


In [26]:
# 4. Load the project's generated analysis tables
monthly = load_table("monthly_kpis.csv", "monthly_kpi.csv")
city = load_table("city_kpis.csv", "city_performance_kpis.csv")
station = load_table("station_kpis.csv", "station_performance_kpis.csv")
rider_activity = load_table("rider_activity_metrics.csv")
rider_segments = load_table("rider_segment_summary.csv")
monthly_active = load_table("monthly_active_riders.csv")
cohort = load_table("rider_cohort_retention.csv")
churn = load_table("rider_churn_summary.csv")
battery_overall = load_table("battery_health_overall.csv")
battery_pack = load_table("battery_health_by_pack_type.csv")
battery_distribution = load_table("battery_health_distribution.csv")
failure_city = load_table("failure_by_city.csv", "city_failure_rates.csv")
failure_station = load_table("failure_by_station.csv", "station_failure_rates.csv")
failure_monthly = load_table("failure_by_month.csv", "monthly_failure_rates.csv")

# Optional tables used by the extended sections.
queue = load_table(
    "queue_metrics.csv", "queue_analysis.csv", "queue_wait_metrics.csv",
    "rider_queue_metrics.csv", "swap_queue_metrics.csv"
)
battery_supplier = load_table(
    "battery_health_by_supplier.csv", "battery_supplier_health.csv",
    "soh_by_supplier.csv"
)
battery_lot = load_table(
    "battery_health_by_lot.csv", "battery_manufacturing_lot_health.csv",
    "soh_by_manufacturing_lot.csv"
)
battery_lifecycle = load_table(
    "battery_lifecycle.csv", "battery_degradation.csv",
    "battery_retirement.csv", "battery_usage_lifecycle.csv"
)
financial = load_table(
    "station_financials.csv", "financial_kpis.csv", "operational_financial_kpis.csv",
    "station_costs.csv", "energy_costs.csv"
)

# Optional raw/source tables for validation and map analysis.
stations_raw = load_optional_raw("stations.csv", "station.csv")
swaps_raw = load_optional_raw("swaps.csv", "swap_events.csv", "swap_transactions.csv")
battery_raw = load_optional_raw("battery_telemetry.csv", "battery_data.csv", "battery_events.csv")
queue_raw = load_optional_raw("queue_events.csv", "queue_event.csv", "swap_queue_events.csv")

loaded_tables = {
    "Monthly KPIs": monthly, "City KPIs": city, "Station KPIs": station,
    "Rider activity": rider_activity, "Rider segments": rider_segments,
    "Monthly active riders": monthly_active, "Cohort retention": cohort,
    "Rider churn": churn, "Battery overall": battery_overall,
    "Battery by pack": battery_pack, "Battery distribution": battery_distribution,
    "Failure by city": failure_city, "Failure by station": failure_station,
    "Failure by month": failure_monthly, "Queue": queue,
    "Battery supplier": battery_supplier, "Battery lot": battery_lot,
    "Battery lifecycle": battery_lifecycle, "Financial": financial,
}
summary = pd.DataFrame([
    {"Table": name, "Rows": len(df), "Columns": len(df.columns), "Available": not df.empty}
    for name, df in loaded_tables.items()
])
display(summary)


Not found: ('monthly_kpis.csv', 'monthly_kpi.csv')
Not found: ('city_kpis.csv', 'city_performance_kpis.csv')
Not found: ('station_kpis.csv', 'station_performance_kpis.csv')
Loaded rider_activity_metrics.csv: 19,950 rows
Loaded rider_segment_summary.csv: 5 rows
Loaded monthly_active_riders.csv: 18 rows
Loaded rider_cohort_retention.csv: 171 rows
Loaded rider_churn_summary.csv: 3 rows
Loaded battery_health_overall.csv: 1 rows
Loaded battery_health_by_pack_type.csv: 2 rows
Loaded battery_health_distribution.csv: 2 rows
Not found: ('failure_by_city.csv', 'city_failure_rates.csv')
Not found: ('failure_by_station.csv', 'station_failure_rates.csv')
Not found: ('failure_by_month.csv', 'monthly_failure_rates.csv')
Not found: ('queue_metrics.csv', 'queue_analysis.csv', 'queue_wait_metrics.csv', 'rider_queue_metrics.csv', 'swap_queue_metrics.csv')
Loaded battery_health_by_supplier.csv: 3 rows
Not found: ('battery_health_by_lot.csv', 'battery_manufacturing_lot_health.csv', 'soh_by_manufacturing_lo

,Table,Rows,Columns,Available
0,Monthly KPIs,0,0,False
1,City KPIs,0,0,False
2,Station KPIs,0,0,False
3,Rider activity,19950,11,True
4,Rider segments,5,7,True
5,Monthly active riders,18,6,True
6,Cohort retention,171,5,True
7,Rider churn,3,6,True
8,Battery overall,1,9,True
9,Battery by pack,2,7,True


In [27]:
# 5. Data-quality summary
quality_rows = []
for name, df in loaded_tables.items():
    if df.empty:
        continue
    missing = int(df.isna().sum().sum())
    dup_rows = int(df.duplicated().sum())
    quality_rows.append({
        "Table": name,
        "Rows": len(df),
        "Columns": len(df.columns),
        "Missing cells": missing,
        "Missing % of cells": round(missing / max(df.size, 1) * 100, 2),
        "Duplicate rows": dup_rows,
    })

quality_summary = pd.DataFrame(quality_rows).sort_values("Missing cells", ascending=False)
display(quality_summary)

# Inspect likely identifier uniqueness.
id_checks = []
for name, df in loaded_tables.items():
    if df.empty:
        continue
    for col in df.columns:
        c = str(col).lower()
        if c == "id" or c.endswith("_id") or c in {"station", "station_id", "rider_id", "battery_id"}:
            non_null = df[col].dropna()
            id_checks.append({
                "Table": name,
                "Identifier": col,
                "Non-null": len(non_null),
                "Unique": non_null.nunique(),
                "Duplicate identifier values": int(non_null.duplicated().sum())
            })
display(pd.DataFrame(id_checks) if id_checks else pd.DataFrame(
    [{"Status": "No obvious identifier columns found in loaded analysis tables."}]
))


,Table,Rows,Columns,Missing cells,Missing % of cells,Duplicate rows
0,Rider activity,19950,11,4,0.00,0
2,Monthly active riders,18,6,2,1.85,0
4,Rider churn,3,6,1,5.56,0
1,Rider segments,5,7,0,0.00,0
3,Cohort retention,171,5,0,0.00,0
5,Battery overall,1,9,0,0.00,0
6,Battery by pack,2,7,0,0.00,0
7,Battery distribution,2,5,0,0.00,0
8,Battery supplier,3,6,0,0.00,0


,Table,Identifier,Non-null,Unique,Duplicate identifier values
0,Rider activity,rider_id,19950,19950,0


In [28]:
# 6. Telemetry, odometer and timestamp validation where source fields exist
validation_rows = []

sources_for_validation = {
    "Swaps/source events": swaps_raw,
    "Battery telemetry": battery_raw,
    "Queue events": queue_raw,
    "Stations": stations_raw,
}
for name, df in sources_for_validation.items():
    if df.empty:
        continue

    row = {"Source": name, "Rows": len(df)}
    for col in df.columns:
        lc = str(col).lower()
        s = pd.to_numeric(df[col], errors="coerce")
        if "odometer" in lc or "distance" in lc:
            row["Negative odometer/distance"] = int((s < 0).sum())
        if "soc" in lc:
            row["SOC outside 0-100"] = int(((s < 0) | (s > 100)).sum())
        if "soh" in lc:
            row["SOH outside 0-100"] = int(((s < 0) | (s > 100)).sum())
        if "latitude" in lc:
            row["Invalid latitude"] = int(((s < -90) | (s > 90)).sum())
        if "longitude" in lc:
            row["Invalid longitude"] = int(((s < -180) | (s > 180)).sum())

    date_cols = [c for c in df.columns if any(k in str(c).lower() for k in ["timestamp", "datetime", "date", "time"])]
    for col in date_cols[:3]:
        parsed = pd.to_datetime(df[col], errors="coerce")
        row[f"{col} parse failures"] = int(parsed.isna().sum())
        if parsed.notna().any():
            row[f"{col} future timestamps"] = int((parsed > pd.Timestamp.now()).sum())

    validation_rows.append(row)

validation_summary = pd.DataFrame(validation_rows)
display(validation_summary if not validation_summary.empty else pd.DataFrame(
    [{"Status": "No raw telemetry/event tables were found; validation is limited to generated analysis tables."}]
))

# Firmware timestamp issue check, if the project contains a firmware-related field.
firmware_checks = []
for name, df in sources_for_validation.items():
    if df.empty:
        continue
    fw_cols = [c for c in df.columns if "firmware" in str(c).lower()]
    time_cols = [c for c in df.columns if any(k in str(c).lower() for k in ["timestamp", "datetime"])]
    if fw_cols and time_cols:
        firmware_checks.append({
            "Source": name,
            "Firmware fields": ", ".join(map(str, fw_cols)),
            "Timestamp fields": ", ".join(map(str, time_cols)),
            "Note": "Review firmware-specific timestamp patterns against the project's known firmware timestamp issue."
        })
display(pd.DataFrame(firmware_checks) if firmware_checks else pd.DataFrame(
    [{"Status": "No firmware/timestamp pair was available for an automated cross-check."}]
))


,Source,Rows,SOC outside 0-100,SOH outside 0-100,Invalid latitude,Invalid longitude,commissioned_date parse failures,commissioned_date future timestamps,decommissioned_date parse failures,decommissioned_date future timestamps,firmware_updated_date parse failures,firmware_updated_date future timestamps
0,Swaps/source events,3877013,3698.0,2799.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Stations,152,NaN,NaN,0.0,0.0,0.0,0.0,151.0,0.0,0.0,0.0


,Status
0,No firmware/timestamp pair was available for a...


## 3. Executive KPI overview

The KPI panel uses the latest dated observation available in the generated monthly/activity tables. It does not invent values when a metric is absent.


In [29]:
# 7. Executive KPIs
mrow = last_row(monthly)
arow = last_row(monthly_active)

kpis = [
    ("Latest completed swaps", value_from(monthly, ["completed_swaps", "total_completed_swaps", "completed_swap_count"], mrow)),
    ("Latest completed swap revenue (INR)", value_from(monthly, ["completed_revenue_inr", "revenue_inr", "total_revenue_inr", "completed_revenue"], mrow)),
    ("Latest monthly active riders", value_from(monthly_active, ["active_riders", "monthly_active_riders"], arow)),
    ("Latest failure rate (%)", value_from(monthly, ["failure_rate_pct", "failure_rate", "failure_pct"], mrow)),
]
kpi_df = pd.DataFrame(kpis, columns=["Metric", "Value"])
kpi_df["Display"] = kpi_df.apply(
    lambda r: f"₹{r['Value']:,.2f}" if pd.notna(r["Value"]) and "INR" in r["Metric"]
    else f"{r['Value']:,.2f}" if pd.notna(r["Value"]) else "Not available",
    axis=1
)
display(kpi_df[["Metric", "Display"]])


,Metric,Display
0,Latest completed swaps,Not available
1,Latest completed swap revenue (INR),Not available
2,Latest monthly active riders,"12,012.00"
3,Latest failure rate (%),Not available


## 4. Network, city and station performance

### 4.1 Monthly network and revenue trends


In [30]:
# 8. Monthly network and revenue trends
if not monthly.empty:
    dc = find_col(monthly, ["month", "year_month", "month_start", "date"])
    sc = find_col(monthly, ["completed_swaps", "total_completed_swaps", "completed_swap_count"])
    rc = find_col(monthly, ["completed_revenue_inr", "revenue_inr", "total_revenue_inr", "completed_revenue"])
    tmp = monthly.copy()
    if dc:
        tmp[dc] = pd.to_datetime(tmp[dc], errors="coerce")
        tmp = tmp.dropna(subset=[dc]).sort_values(dc)
        if sc:
            fig = px.line(tmp, x=dc, y=sc, markers=True, title="Monthly Completed Swaps")
            fig.update_layout(template="plotly_white", xaxis_title="Month", yaxis_title="Completed swaps")
            fig.show()
        if rc:
            fig = px.line(tmp, x=dc, y=rc, markers=True, title="Monthly Completed Swap Revenue (INR)")
            fig.update_layout(template="plotly_white", yaxis_tickprefix="₹", yaxis_tickformat=",.0f")
            fig.show()
    else:
        print("No month/date field found.")
else:
    print("Monthly KPI table is unavailable.")


Monthly KPI table is unavailable.


### 4.2 City performance


In [31]:
# 9. City performance
if not city.empty:
    cc = find_col(city, ["city", "city_name"])
    if cc:
        numeric = city.select_dtypes(include="number").columns.tolist()
        cols = [c for c in numeric if any(k in str(c).lower() for k in ["revenue", "completed", "failure", "rate", "swap"])]
        for col in cols[:5]:
            fig = px.bar(city.sort_values(col, ascending=False), x=cc, y=col,
                         title=f"City comparison — {col}", text_auto=".3s")
            fig.update_layout(template="plotly_white")
            fig.show()
    else:
        display(city.head())
else:
    print("City KPI table is unavailable.")


City KPI table is unavailable.


### 4.3 Station performance


In [32]:
# 10. Station performance
if not station.empty:
    sid = find_col(station, ["station_id", "station", "station_name"])
    if sid:
        numeric = station.select_dtypes(include="number").columns.tolist()
        cols = [c for c in numeric if any(k in str(c).lower() for k in ["completed", "revenue", "failure", "rate", "swap"])]
        for col in cols[:4]:
            top = station.sort_values(col, ascending=False).head(15)
            fig = px.bar(top, x=col, y=sid, orientation="h",
                         title=f"Top 15 stations by {col}", text_auto=".3s")
            fig.update_layout(template="plotly_white", yaxis={"categoryorder":"total ascending"})
            fig.show()
    else:
        display(station.head())
else:
    print("Station KPI table is unavailable.")


Station KPI table is unavailable.


### 4.4 Station location map

If `stations.csv` contains latitude/longitude, marker size represents an available station-level volume metric and hover text provides station identity. If completed swaps/failure rate are available in the station KPI table, they are merged for context.


In [33]:
# 11. Station location map
if not stations_raw.empty:
    lat = find_col(stations_raw, ["latitude", "lat"])
    lon = find_col(stations_raw, ["longitude", "lon", "lng"])
    sid_raw = find_col(stations_raw, ["station_id", "station", "station_name"])
    if lat and lon:
        map_df = stations_raw.copy()
        map_df[lat] = pd.to_numeric(map_df[lat], errors="coerce")
        map_df[lon] = pd.to_numeric(map_df[lon], errors="coerce")
        map_df = map_df.dropna(subset=[lat, lon])

        if not station.empty and sid_raw:
            sid_station = find_col(station, ["station_id", "station", "station_name"])
            if sid_station:
                merge_cols = [sid_station]
                volume_col = numeric_col(station, ["completed_swaps", "total_completed_swaps", "completed_swap_count", "swap_count"])
                failure_col = numeric_col(station, ["failure_rate_pct", "failure_rate", "failure_pct"])
                if volume_col: merge_cols.append(volume_col)
                if failure_col: merge_cols.append(failure_col)
                station_context = station[merge_cols].drop_duplicates(subset=[sid_station]).copy()
                station_context = station_context.rename(columns={sid_station: sid_raw})
                map_df = map_df.merge(station_context, on=sid_raw, how="left")

        size_col = numeric_col(map_df, ["completed_swaps", "total_completed_swaps", "completed_swap_count", "swap_count"])
        color_col = numeric_col(map_df, ["failure_rate_pct", "failure_rate", "failure_pct"])

        kwargs = dict(
            data_frame=map_df, lat=lat, lon=lon,
            hover_name=sid_raw, title="VoltRelay Station Locations"
        )
        if size_col:
            kwargs["size"] = size_col
        if color_col:
            kwargs["color"] = color_col
            kwargs["color_continuous_scale"] = "Reds"

        fig = px.scatter_mapbox(**kwargs)
        fig.update_layout(mapbox_style="open-street-map", height=650, margin=dict(l=0,r=0,t=50,b=0))
        fig.show()
    else:
        print("stations.csv was found, but latitude/longitude fields were not identified.")
else:
    print("Station location map skipped: stations.csv was not found.")


C:\Users\singi\AppData\Local\Temp\ipykernel_9020\2504514070.py:37: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(**kwargs)


## 5. Reliability, failures and queue analysis

Failure rate in the existing project excludes abandoned and cancelled attempts where that definition is provided by the generated failure tables. Queue analysis is descriptive: a relationship between waiting time and abandonment does **not** establish that waiting caused abandonment.


In [34]:
# 12. Failure analysis
if not failure_city.empty:
    display(failure_city.head(10))
    cc = find_col(failure_city, ["city", "city_name"])
    rc = find_col(failure_city, ["failure_rate_pct", "failure_rate", "failure_pct"])
    if cc and rc:
        fig = px.bar(failure_city.sort_values(rc, ascending=False), x=cc, y=rc,
                     title="Failure Rate by City", text_auto=".2f")
        fig.update_layout(template="plotly_white", yaxis_title="Failure rate (%)")
        fig.show()

if not failure_monthly.empty:
    dc = find_col(failure_monthly, ["month", "year_month", "date"])
    rc = find_col(failure_monthly, ["failure_rate_pct", "failure_rate", "failure_pct"])
    if dc and rc:
        tmp = failure_monthly.copy()
        tmp[dc] = pd.to_datetime(tmp[dc], errors="coerce")
        tmp = tmp.dropna(subset=[dc]).sort_values(dc)
        fig = px.line(tmp, x=dc, y=rc, markers=True, title="Monthly Failure Rate")
        fig.update_layout(template="plotly_white", yaxis_title="Failure rate (%)")
        fig.show()

if not failure_station.empty:
    display(failure_station.head(15))


### 5.1 Queue waiting time and abandoned swaps


In [35]:
# 13. Queue waiting time and abandonment
queue_df = queue if not queue.empty else queue_raw

if not queue_df.empty:
    wait_col = numeric_col(queue_df, [
        "queue_wait_minutes", "wait_minutes", "waiting_minutes",
        "queue_wait_time", "wait_time_minutes", "queue_duration_minutes"
    ])
    status_col = find_col(queue_df, ["status", "queue_status", "swap_status", "outcome"])
    city_col = find_col(queue_df, ["city", "city_name"])
    station_col = find_col(queue_df, ["station_id", "station", "station_name"])

    if wait_col:
        wait = pd.to_numeric(queue_df[wait_col], errors="coerce").dropna()
        wait = wait[wait >= 0]
        if len(wait):
            wait_summary = pd.DataFrame([{
                "Mean wait (min)": wait.mean(),
                "Median wait (min)": wait.median(),
                "90th percentile wait (min)": wait.quantile(.90),
                "Observed queue records": len(wait)
            }])
            display(wait_summary.round(2))

        if status_col:
            status = queue_df[status_col].astype(str).str.lower().str.strip()
            abandoned = status.str.contains("abandon|cancel", regex=True, na=False)
            q = queue_df.assign(_abandoned=abandoned)
            if city_col:
                city_abandon = q.groupby(city_col)["_abandoned"].agg(["sum", "count"]).reset_index()
                city_abandon["abandonment_rate_pct"] = pct(city_abandon["sum"], city_abandon["count"])
                display(city_abandon.sort_values("abandonment_rate_pct", ascending=False).head(15))

            if station_col:
                station_abandon = q.groupby(station_col)["_abandoned"].agg(["sum", "count"]).reset_index()
                station_abandon["abandonment_rate_pct"] = pct(station_abandon["sum"], station_abandon["count"])
                display(station_abandon.sort_values("abandonment_rate_pct", ascending=False).head(15))

            plot_df = q[[wait_col, "_abandoned"]].copy()
            plot_df[wait_col] = pd.to_numeric(plot_df[wait_col], errors="coerce")
            plot_df = plot_df.dropna(subset=[wait_col])
            if len(plot_df) > 0:
                bins = [-np.inf, 2, 5, 10, 20, 30, np.inf]
                labels = ["≤2", "2–5", "5–10", "10–20", "20–30", "30+"]
                plot_df["Wait band (min)"] = pd.cut(plot_df[wait_col], bins=bins, labels=labels)
                band = plot_df.groupby("Wait band (min)", observed=False)["_abandoned"].mean().reset_index()
                band["abandonment_rate_pct"] = band["_abandoned"] * 100
                fig = px.bar(band, x="Wait band (min)", y="abandonment_rate_pct",
                             title="Abandonment rate by queue-wait band")
                fig.update_layout(template="plotly_white", yaxis_title="Abandonment rate (%)")
                fig.show()
                display(Markdown("**Interpretation:** This chart describes an observed association between wait bands and abandonment; it does not establish causation."))
    else:
        print("Queue data was found, but no waiting-time field was identified.")
else:
    print("Queue analysis skipped: no queue-event/queue-metric table was found.")


Queue analysis skipped: no queue-event/queue-metric table was found.


## 6. Battery health and lifecycle

This section extends the existing SOH analysis with:
- health distribution;
- pack-type comparison;
- supplier and manufacturing-lot comparisons when available;
- utilization/degradation/retirement patterns when lifecycle fields exist.


In [36]:
# 14. Battery health overview and pack type
if not battery_overall.empty:
    display(battery_overall)

if not battery_pack.empty:
    display(battery_pack)
    pc = find_col(battery_pack, ["pack_type", "battery_type", "type"])
    soh = numeric_col(battery_pack, ["avg_current_soh_pct", "average_current_soh_pct", "current_soh_pct", "avg_soh_pct"])
    if pc and soh:
        fig = px.bar(battery_pack, x=pc, y=soh,
                     title="Average Current Battery SOH by Pack Type", text_auto=".2f")
        fig.update_layout(template="plotly_white", yaxis_title="Average current SOH (%)")
        fig.show()

if not battery_distribution.empty:
    display(battery_distribution)
    cat = find_col(battery_distribution, ["health_category", "category", "soh_category"])
    cnt = numeric_col(battery_distribution, ["battery_count", "count", "total_batteries"])
    if cat and cnt:
        fig = px.bar(battery_distribution, x=cat, y=cnt,
                     title="Battery Count by Health Category", text_auto=True)
        fig.update_layout(template="plotly_white")
        fig.show()


,total_batteries,valid_current_soh,average_initial_soh_pct,average_current_soh_pct,average_reported_degradation_pct,batteries_below_80_soh,batteries_below_70_soh,retired_batteries,average_age_days
0,6500,6500,99.191292,76.614585,22.576708,3502,1438,1438,349.896462


,pack_type,battery_count,average_initial_soh_pct,average_current_soh_pct,average_degradation_pct,average_age_days,retired_batteries
0,2W_2.1kWh,5518,99.187677,75.088909,24.098768,354.189199,1385
1,3W_4.8kWh,982,99.211609,85.187576,14.024033,325.774949,53


,health_category,battery_count,average_current_soh_pct,average_initial_soh_pct,average_age_days
0,Aging (70-79.9%),5062,81.100277,99.189925,381.252667
1,Critical (<70%),1438,60.824200,99.196106,239.517385


In [37]:
# 15. Battery supplier and manufacturing-lot comparisons
for label, df, group_candidates in [
    ("Supplier", battery_supplier, ["supplier", "supplier_name", "battery_supplier"]),
    ("Manufacturing lot", battery_lot, ["lot", "lot_id", "manufacturing_lot", "batch", "batch_id"]),
]:
    if df.empty:
        print(f"{label} analysis skipped: source table unavailable.")
        continue
    group = find_col(df, group_candidates)
    soh = numeric_col(df, ["avg_current_soh_pct", "average_current_soh_pct", "current_soh_pct", "avg_soh_pct", "soh_pct"])
    if group and soh:
        tmp = df[[group, soh]].copy().sort_values(soh)
        display(tmp)
        fig = px.bar(tmp, x=group, y=soh, title=f"Average SOH by {label}",
                     text_auto=".2f")
        fig.update_layout(template="plotly_white", yaxis_title="SOH (%)")
        fig.show()
    else:
        display(df.head())
        print(f"{label} table found, but matching group/SOH fields were not identified.")


,supplier,average_current_soh_pct
2,Kyron,63.747436
1,Cellora,80.929030
0,Amptek,81.005203


Manufacturing lot analysis skipped: source table unavailable.


In [38]:
# 16. Battery utilization, degradation and retirement patterns
if not battery_lifecycle.empty:
    display(battery_lifecycle.head(10))
    date_col = find_col(battery_lifecycle, ["date", "month", "year_month", "retirement_date"])
    usage_col = numeric_col(battery_lifecycle, ["usage_count", "swap_count", "cycle_count", "swaps_completed"])
    soh_col = numeric_col(battery_lifecycle, ["soh_pct", "current_soh_pct", "avg_soh_pct", "average_soh_pct"])
    type_col = find_col(battery_lifecycle, ["pack_type", "battery_type", "type"])

    if usage_col and soh_col:
        fig = px.scatter(
            battery_lifecycle, x=usage_col, y=soh_col,
            color=type_col if type_col else None,
            trendline="ols" if len(battery_lifecycle.dropna(subset=[usage_col, soh_col])) >= 3 else None,
            title="Battery Utilization vs SOH"
        )
        fig.update_layout(template="plotly_white", xaxis_title="Usage / cycles", yaxis_title="SOH (%)")
        fig.show()

    if date_col and soh_col:
        tmp = battery_lifecycle.copy()
        tmp[date_col] = pd.to_datetime(tmp[date_col], errors="coerce")
        tmp = tmp.dropna(subset=[date_col])
        if len(tmp):
            trend = tmp.groupby(pd.Grouper(key=date_col, freq="MS"))[soh_col].mean().reset_index()
            fig = px.line(trend, x=date_col, y=soh_col, markers=True,
                          title="Average Battery SOH Over Lifecycle Time")
            fig.update_layout(template="plotly_white", yaxis_title="Average SOH (%)")
            fig.show()
else:
    print("Battery lifecycle analysis skipped: no lifecycle/degradation/retirement table was found.")


Battery lifecycle analysis skipped: no lifecycle/degradation/retirement table was found.


## 7. Rider engagement, retention and churn


In [39]:
# 17. Rider activity, retention and cohort heatmap
if not monthly_active.empty:
    dc = find_col(monthly_active, ["month", "year_month", "date"])
    ac = numeric_col(monthly_active, ["active_riders", "monthly_active_riders"])
    if dc and ac:
        tmp = monthly_active.copy()
        tmp[dc] = pd.to_datetime(tmp[dc], errors="coerce")
        tmp = tmp.dropna(subset=[dc]).sort_values(dc)
        fig = px.line(tmp, x=dc, y=ac, markers=True, title="Monthly Active Riders")
        fig.update_layout(template="plotly_white", yaxis_title="Active riders")
        fig.show()

if not rider_segments.empty:
    display(rider_segments)
    seg = find_col(rider_segments, ["segment"])
    cnt = numeric_col(rider_segments, ["rider_count", "count", "riders"])
    if seg and cnt:
        fig = px.bar(rider_segments.sort_values(cnt), x=cnt, y=seg, orientation="h",
                     title="Riders by Engagement Segment", text_auto=True)
        fig.update_layout(template="plotly_white")
        fig.show()

if not cohort.empty:
    cc = find_col(cohort, ["cohort_month"])
    age = numeric_col(cohort, ["month_number", "months_since_cohort", "cohort_age_months"])
    ret = numeric_col(cohort, ["retention_rate_pct", "retention_rate", "retention_pct", "retention_percentage"])
    if cc and age and ret:
        tmp = cohort.copy()
        tmp[cc] = pd.to_datetime(tmp[cc], errors="coerce").dt.strftime("%Y-%m")
        tmp[age] = pd.to_numeric(tmp[age], errors="coerce")
        tmp[ret] = pd.to_numeric(tmp[ret], errors="coerce")
        pivot = tmp.pivot_table(index=cc, columns=age, values=ret, aggfunc="mean")
        fig = px.imshow(pivot, aspect="auto", color_continuous_scale="Blues",
                        title="Rider Cohort Retention (%)",
                        labels={"x":"Months since cohort start", "y":"Cohort month", "color":"Retention (%)"})
        fig.update_layout(template="plotly_white")
        fig.show()
    else:
        display(cohort.head())

if not churn.empty:
    display(churn)


,rider_segment,rider_count,completed_swaps,average_completed_swaps,total_revenue_inr,average_revenue_inr,rider_share_pct
0,Frequent Rider (21+),17732,3622633.0,204.299177,2.315393e+08,13057.707585,88.88
1,No Completed Swaps,2,0.0,0.000000,0.000000e+00,0.000000,0.01
2,Occasional Rider (2-5),432,1522.0,3.523148,1.043289e+05,241.502083,2.17
3,One-Time Rider,103,103.0,1.000000,6.998400e+03,67.945631,0.52
4,Regular Rider (6-20),1681,21551.0,12.820345,1.475457e+06,877.725967,8.43


,churn_status,rider_count,completed_swaps,total_revenue_inr,average_completed_swaps,average_days_since_last_swap
0,Active in Final 30 Days,12075,2561206.0,1.645943e+08,212.108157,2.094410
1,Inactive in Final 30 Days,7873,1084603.0,6.853175e+07,137.762352,224.530801
2,No Completed Swap,2,0.0,0.000000e+00,0.000000,NaN


## 8. Financial and operational efficiency

Where the project provides the required fields, the notebook calculates:
- revenue per completed swap;
- grid energy cost from kWh × tariff;
- energy cost per completed swap;
- station-level revenue/energy-efficiency indicators.

**Accounting caution:** revenue minus electricity cost is not labeled profit because operating, battery, rent, maintenance, partner, and other costs may be missing.


In [40]:
# 18. Financial and operational efficiency
def show_efficiency_from_monthly(df):
    if df.empty:
        return
    swaps = numeric_col(df, ["completed_swaps", "total_completed_swaps", "completed_swap_count"])
    revenue = numeric_col(df, ["completed_revenue_inr", "revenue_inr", "total_revenue_inr", "completed_revenue"])
    kwh = numeric_col(df, ["grid_kwh", "energy_kwh", "electricity_kwh", "grid_energy_kwh"])
    tariff = numeric_col(df, ["grid_tariff", "tariff_inr_per_kwh", "electricity_tariff"])
    if swaps and revenue:
        eff = df.copy()
        eff["revenue_per_completed_swap_inr"] = np.where(
            pd.to_numeric(eff[swaps], errors="coerce") != 0,
            pd.to_numeric(eff[revenue], errors="coerce") / pd.to_numeric(eff[swaps], errors="coerce"),
            np.nan
        )
        cols = [c for c in [find_col(eff, ["month","year_month","date"]), swaps, revenue, "revenue_per_completed_swap_inr"] if c]
        if kwh:
            eff["grid_kwh_numeric"] = pd.to_numeric(eff[kwh], errors="coerce")
            cols.append("grid_kwh_numeric")
        if kwh and tariff:
            eff["energy_cost_inr"] = (
                pd.to_numeric(eff[kwh], errors="coerce") *
                pd.to_numeric(eff[tariff], errors="coerce")
            )
            eff["energy_cost_per_completed_swap_inr"] = np.where(
                pd.to_numeric(eff[swaps], errors="coerce") != 0,
                eff["energy_cost_inr"] / pd.to_numeric(eff[swaps], errors="coerce"),
                np.nan
            )
            cols += ["energy_cost_inr", "energy_cost_per_completed_swap_inr"]
        display(eff[cols].tail(15).round(2))

show_efficiency_from_monthly(monthly)
show_efficiency_from_monthly(financial)


In [41]:
# 19. Station-level efficiency, when station financial fields are available
eff_source = financial if not financial.empty else station
if not eff_source.empty:
    station_id = find_col(eff_source, ["station_id", "station", "station_name"])
    swaps = numeric_col(eff_source, ["completed_swaps", "total_completed_swaps", "completed_swap_count"])
    revenue = numeric_col(eff_source, ["completed_revenue_inr", "revenue_inr", "total_revenue_inr", "completed_revenue"])
    kwh = numeric_col(eff_source, ["grid_kwh", "energy_kwh", "electricity_kwh", "grid_energy_kwh"])
    if station_id and swaps and revenue:
        tmp = eff_source.copy()
        tmp["revenue_per_completed_swap_inr"] = np.where(
            pd.to_numeric(tmp[swaps], errors="coerce") != 0,
            pd.to_numeric(tmp[revenue], errors="coerce") / pd.to_numeric(tmp[swaps], errors="coerce"),
            np.nan
        )
        display(tmp[[station_id, swaps, revenue, "revenue_per_completed_swap_inr"]].sort_values(
            "revenue_per_completed_swap_inr", ascending=False
        ).head(15).round(2))
    if station_id and swaps and kwh:
        tmp = eff_source.copy()
        tmp["kwh_per_completed_swap"] = np.where(
            pd.to_numeric(tmp[swaps], errors="coerce") != 0,
            pd.to_numeric(tmp[kwh], errors="coerce") / pd.to_numeric(tmp[swaps], errors="coerce"),
            np.nan
        )
        display(tmp[[station_id, swaps, kwh, "kwh_per_completed_swap"]].sort_values(
            "kwh_per_completed_swap", ascending=True
        ).head(15).round(3))
else:
    print("Station efficiency skipped: no financial/station source was available.")


Station efficiency skipped: no financial/station source was available.


## 9. Business insights and recommendations

The recommendations below are framed as operational hypotheses or experiments. They should be tested against future observations rather than treated as causal conclusions.


In [42]:
# 20. Evidence-aware business recommendations
recommendations = [
    ("Rider reactivation",
     "Evaluate targeted campaigns for riders classified inactive; measure subsequent completed swaps and use a comparison group where feasible."),
    ("Retention monitoring",
     "Compare cohorts at the same cohort age because newer cohorts have shorter observation windows."),
    ("Queue experience",
     "Monitor median and 90th-percentile waiting time and investigate stations/cities with higher observed abandonment."),
    ("Reliability operations",
     "Investigate higher-failure stations alongside inventory, outage, quarantine, and telemetry context. Observed associations are not proof of cause."),
    ("Battery lifecycle",
     "Track SOH by pack type, supplier and manufacturing lot where available; investigate high-degradation groups before changing procurement or retirement policy."),
    ("Energy efficiency",
     "Track kWh per completed swap and energy cost per completed swap where meter and tariff data are available."),
    ("Revenue interpretation",
     "Completed swap charges are revenue, not profit. Add operating, battery depreciation, discounts, partner payments, maintenance and station costs for profitability analysis."),
    ("Privacy",
     "Use aggregated rider-level outputs in presentations and avoid publishing rider identifiers.")
]
display(pd.DataFrame(recommendations, columns=["Area", "Recommendation"]))


,Area,Recommendation
0,Rider reactivation,Evaluate targeted campaigns for riders classif...
1,Retention monitoring,Compare cohorts at the same cohort age because...
2,Queue experience,Monitor median and 90th-percentile waiting tim...
3,Reliability operations,Investigate higher-failure stations alongside ...
4,Battery lifecycle,"Track SOH by pack type, supplier and manufactu..."
5,Energy efficiency,Track kWh per completed swap and energy cost p...
6,Revenue interpretation,"Completed swap charges are revenue, not profit..."
7,Privacy,Use aggregated rider-level outputs in presenta...


In [43]:
# 21. Five findings calculated from the available outputs
findings = []

if not monthly.empty:
    swaps = numeric_col(monthly, ["completed_swaps", "total_completed_swaps", "completed_swap_count"])
    revenue = numeric_col(monthly, ["completed_revenue_inr", "revenue_inr", "total_revenue_inr", "completed_revenue"])
    date_col = find_col(monthly, ["month", "year_month", "month_start", "date"])
    tmp = monthly.copy()
    if date_col:
        tmp[date_col] = pd.to_datetime(tmp[date_col], errors="coerce")
        tmp = tmp.sort_values(date_col)
    if swaps and len(tmp) >= 2:
        s = pd.to_numeric(tmp[swaps], errors="coerce")
        findings.append(("Latest vs first-period completed swaps",
                         f"{s.iloc[-1]:,.0f} versus {s.iloc[0]:,.0f}" if s.notna().iloc[-1] and s.notna().iloc[0] else "Not available"))
    if revenue and len(tmp) >= 2:
        r = pd.to_numeric(tmp[revenue], errors="coerce")
        findings.append(("Latest vs first-period completed revenue",
                         f"₹{r.iloc[-1]:,.0f} versus ₹{r.iloc[0]:,.0f}" if r.notna().iloc[-1] and r.notna().iloc[0] else "Not available"))

if not failure_city.empty:
    city_name = find_col(failure_city, ["city", "city_name"])
    fr = numeric_col(failure_city, ["failure_rate_pct", "failure_rate", "failure_pct"])
    if city_name and fr:
        top = failure_city.loc[pd.to_numeric(failure_city[fr], errors="coerce").idxmax()]
        findings.append(("Highest observed city failure rate",
                         f"{top[city_name]} — {float(top[fr]):.2f}%"))

if not battery_pack.empty:
    pack = find_col(battery_pack, ["pack_type", "battery_type", "type"])
    soh = numeric_col(battery_pack, ["avg_current_soh_pct", "average_current_soh_pct", "current_soh_pct", "avg_soh_pct"])
    if pack and soh:
        top = battery_pack.loc[pd.to_numeric(battery_pack[soh], errors="coerce").idxmin()]
        findings.append(("Lowest observed average pack SOH",
                         f"{top[pack]} — {float(top[soh]):.2f}%"))

if not monthly_active.empty:
    active = numeric_col(monthly_active, ["active_riders", "monthly_active_riders"])
    if active:
        a = pd.to_numeric(monthly_active[active], errors="coerce").dropna()
        if len(a):
            findings.append(("Peak monthly active riders", f"{a.max():,.0f}"))

display(pd.DataFrame(findings, columns=["Finding", "Evidence"]) if findings else
        pd.DataFrame([{"Finding": "No calculable findings", "Evidence": "Required source fields were not available."}]))


,Finding,Evidence
0,Lowest observed average pack SOH,2W_2.1kWh — 75.09%
1,Peak monthly active riders,"12,012"


## 10. Limitations, conclusion and future scope

### Limitations
- The project dataset is synthetic.
- The notebook depends on generated analysis tables; missing optional source tables cause individual sections to be skipped.
- Rider inactivity is a churn proxy, not confirmed permanent churn.
- Descriptive correlations and associations do not establish causality.
- Revenue is not profit.
- Station maps depend on valid latitude/longitude data.
- Queue and battery-lifecycle conclusions depend on the availability and quality of the relevant event tables.

### Conclusion
The notebook provides an end-to-end decision-support view across network performance, reliability, battery health, rider retention, operational efficiency, and business actions. Before final submission, run the notebook from top to bottom in the actual project environment and confirm that every intended chart renders and every expected CSV is found.

### Future scope
- Add controlled experiments for rider reactivation and retention interventions.
- Build station-level predictive failure and demand models.
- Add battery degradation forecasting using cycle/temperature/SOH history.
- Add a formal cost model for full contribution margin and profitability.
- Monitor queue SLA metrics and alert thresholds in production.


In [44]:
# 22. Export a presentation-ready Excel workbook
export_path = OUTPUT_DIR / "volt_relay_dashboard_summary.xlsx"
export_tables = {
    "Monthly_KPIs": monthly, "City_KPIs": city, "Station_KPIs": station,
    "Failure_City": failure_city, "Failure_Station": failure_station, "Failure_Monthly": failure_monthly,
    "Rider_Segments": rider_segments, "Monthly_Active": monthly_active,
    "Cohort_Retention": cohort, "Rider_Churn": churn,
    "Battery_Overall": battery_overall, "Battery_By_Pack": battery_pack,
    "Battery_Distribution": battery_distribution, "Queue": queue,
    "Battery_Supplier": battery_supplier, "Battery_Lot": battery_lot,
    "Battery_Lifecycle": battery_lifecycle, "Financial": financial,
    "Data_Quality": quality_summary, "Validation": validation_summary,
    "Recommendations": pd.DataFrame(recommendations, columns=["Area", "Recommendation"]),
}
available = {name: df for name, df in export_tables.items() if isinstance(df, pd.DataFrame) and not df.empty}

if available:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    try:
        with pd.ExcelWriter(export_path, engine="openpyxl") as writer:
            for name, df in available.items():
                df.to_excel(writer, sheet_name=name[:31], index=False)
        print("Summary workbook saved:", export_path)
        print("Sheets:", ", ".join(available.keys()))
    except Exception as e:
        print("Excel export skipped:", e)
else:
    print("No tables were loaded; check project paths and generated CSVs.")


Excel export skipped: No module named 'openpyxl'


In [45]:
# 23. Final run-readiness check

expected_sections = [
    "Project overview",
    "Dataset description and data quality",
    "Executive KPI overview",
    "Network, city and station performance",
    "Reliability, failures and queue analysis",
    "Battery health and lifecycle",
    "Rider engagement, retention and churn",
    "Financial and operational efficiency",
    "Business insights and recommendations",
    "Limitations, conclusion and future scope"
]

print("Final run-readiness check")
print("=" * 50)

print("\nExpected sections:")
for i, section in enumerate(expected_sections, 1):
    print(f"{i:02d}. {section}")

print("\nBefore submission:")
print("1. Run Kernel → Restart Kernel and Run All.")
print("2. Confirm there are no red error cells.")
print("3. Confirm station map renders when stations.csv has coordinates.")
print("4. Confirm queue/battery-lifecycle sections use the intended project tables.")
print("5. Confirm the Excel workbook is created in data/outputs.")

print("\nNotebook structure check completed successfully.")

Final run-readiness check

Expected sections:
01. Project overview
02. Dataset description and data quality
03. Executive KPI overview
04. Network, city and station performance
05. Reliability, failures and queue analysis
06. Battery health and lifecycle
07. Rider engagement, retention and churn
08. Financial and operational efficiency
09. Business insights and recommendations
10. Limitations, conclusion and future scope

Before submission:
1. Run Kernel → Restart Kernel and Run All.
2. Confirm there are no red error cells.
3. Confirm station map renders when stations.csv has coordinates.
4. Confirm queue/battery-lifecycle sections use the intended project tables.
5. Confirm the Excel workbook is created in data/outputs.

Notebook structure check completed successfully.
